# Day 4: Monitoring & Production

**MLOps in Practice Workshop**

---

## Learning Objectives

1. Understand why models degrade in production
2. Detect data drift using PSI and the Kolmogorov-Smirnov test
3. Generate drift reports with Evidently AI
4. Design production monitoring patterns
5. Apply MLOps in low-resource environments

---

## 1. Model Monitoring Concepts

A model that performs well at deployment will degrade over time. This is not a possibility -- it is a certainty.

### Why Models Fail in Production

| Cause | Description | Example |
|-------|-------------|---------|
| **Data drift** | Input data distribution changes | Customer demographics shift after market expansion |
| **Concept drift** | The relationship between inputs and outputs changes | What defines a fraudulent transaction evolves |
| **Feature drift** | Individual feature distributions change | Seasonal effects on purchasing patterns |
| **Label drift** | Target variable distribution changes | Default rates change during economic downturn |
| **Upstream changes** | Data pipeline modifications | A vendor changes their data schema |

### What to Monitor

1. **Input data quality** -- missing values, schema violations, out-of-range values
2. **Feature distributions** -- compare incoming data to training data
3. **Prediction distributions** -- sudden changes in output distribution
4. **Model performance** -- accuracy, latency, error rates (when labels are available)
5. **System health** -- CPU, memory, request rate, response time

---

## 2. Data Drift Detection

### 2.1 Population Stability Index (PSI)

PSI measures how much a distribution has shifted between two datasets (e.g., training vs. production).

$$\text{PSI} = \sum_{i=1}^{n} (p_i^{\text{actual}} - p_i^{\text{expected}}) \times \ln\left(\frac{p_i^{\text{actual}}}{p_i^{\text{expected}}}\right)$$

| PSI Value | Interpretation |
|-----------|----------------|
| < 0.1 | No significant change |
| 0.1 -- 0.2 | Moderate change -- investigate |
| > 0.2 | Significant change -- action required |

In [ ]:
# Install dependencies
# !pip install scikit-learn pandas numpy matplotlib evidently

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats


def compute_psi(expected: np.ndarray, actual: np.ndarray, n_bins: int = 10) -> float:
    """
    Compute the Population Stability Index (PSI) between two distributions.

    Parameters
    ----------
    expected : array-like
        Reference distribution (e.g., training data).
    actual : array-like
        Current distribution (e.g., production data).
    n_bins : int
        Number of bins for histogram comparison.

    Returns
    -------
    float
        PSI value.
    """
    # Define bins based on the expected distribution
    breakpoints = np.percentile(expected, np.linspace(0, 100, n_bins + 1))
    breakpoints[0] = -np.inf
    breakpoints[-1] = np.inf

    # Compute proportions in each bin
    expected_counts = np.histogram(expected, bins=breakpoints)[0]
    actual_counts = np.histogram(actual, bins=breakpoints)[0]

    # Convert to proportions with smoothing to avoid division by zero
    expected_prop = (expected_counts + 1e-6) / (expected_counts.sum() + 1e-6 * n_bins)
    actual_prop = (actual_counts + 1e-6) / (actual_counts.sum() + 1e-6 * n_bins)

    # PSI formula
    psi = np.sum((actual_prop - expected_prop) * np.log(actual_prop / expected_prop))
    return psi


print("PSI function defined.")

### 2.2 Kolmogorov-Smirnov Test

The KS test is a nonparametric test that compares two distributions. It returns a test statistic (maximum distance between CDFs) and a p-value.

In [ ]:
def ks_drift_test(reference: np.ndarray, current: np.ndarray, alpha: float = 0.05) -> dict:
    """
    Perform the Kolmogorov-Smirnov test for drift detection.

    Parameters
    ----------
    reference : array-like
        Reference distribution.
    current : array-like
        Current distribution to test.
    alpha : float
        Significance level.

    Returns
    -------
    dict
        Test results with statistic, p-value, and drift flag.
    """
    statistic, p_value = stats.ks_2samp(reference, current)
    return {
        "statistic": statistic,
        "p_value": p_value,
        "drift_detected": p_value < alpha,
    }


print("KS test function defined.")

In [ ]:
# Generate synthetic training and production data
np.random.seed(42)

# Training data -- our reference distribution
n_train = 5000
train_data = pd.DataFrame({
    "age": np.random.normal(35, 10, n_train).clip(18, 80),
    "income": np.random.lognormal(10.5, 0.5, n_train),
    "credit_score": np.random.normal(680, 50, n_train).clip(300, 850),
    "loan_amount": np.random.lognormal(9.5, 0.6, n_train),
})

# Production data -- no drift (similar distribution)
n_prod = 2000
prod_no_drift = pd.DataFrame({
    "age": np.random.normal(35, 10, n_prod).clip(18, 80),
    "income": np.random.lognormal(10.5, 0.5, n_prod),
    "credit_score": np.random.normal(680, 50, n_prod).clip(300, 850),
    "loan_amount": np.random.lognormal(9.5, 0.6, n_prod),
})

# Production data -- WITH drift (shifted distributions)
prod_with_drift = pd.DataFrame({
    "age": np.random.normal(45, 12, n_prod).clip(18, 80),        # older population
    "income": np.random.lognormal(10.8, 0.7, n_prod),             # higher income
    "credit_score": np.random.normal(650, 60, n_prod).clip(300, 850),  # lower scores
    "loan_amount": np.random.lognormal(10.0, 0.8, n_prod),        # larger loans
})

print("Synthetic datasets created:")
print(f"  Training: {train_data.shape}")
print(f"  Production (no drift): {prod_no_drift.shape}")
print(f"  Production (with drift): {prod_with_drift.shape}")

In [ ]:
# Compute PSI and KS test for each feature

features = ["age", "income", "credit_score", "loan_amount"]

print("=" * 70)
print("Scenario 1: No Drift (training vs. stable production)")
print("=" * 70)

for feat in features:
    psi = compute_psi(train_data[feat].values, prod_no_drift[feat].values)
    ks = ks_drift_test(train_data[feat].values, prod_no_drift[feat].values)
    print(f"  {feat:15s} | PSI={psi:.4f} | KS stat={ks['statistic']:.4f} | "
          f"p={ks['p_value']:.4f} | drift={ks['drift_detected']}")

print()
print("=" * 70)
print("Scenario 2: With Drift (training vs. shifted production)")
print("=" * 70)

for feat in features:
    psi = compute_psi(train_data[feat].values, prod_with_drift[feat].values)
    ks = ks_drift_test(train_data[feat].values, prod_with_drift[feat].values)
    psi_flag = "ALERT" if psi > 0.2 else ("WARN" if psi > 0.1 else "OK")
    print(f"  {feat:15s} | PSI={psi:.4f} [{psi_flag:5s}] | KS stat={ks['statistic']:.4f} | "
          f"p={ks['p_value']:.6f} | drift={ks['drift_detected']}")

In [ ]:
# Visualize distributions: training vs. drifted production

fig, axes = plt.subplots(2, 2, figsize=(12, 8))
fig.suptitle("Feature Distributions: Training vs. Drifted Production", fontsize=14)

for ax, feat in zip(axes.flatten(), features):
    ax.hist(train_data[feat], bins=40, alpha=0.5, label="Training", density=True, color="steelblue")
    ax.hist(prod_with_drift[feat], bins=40, alpha=0.5, label="Production (drifted)", density=True, color="coral")

    psi = compute_psi(train_data[feat].values, prod_with_drift[feat].values)
    ax.set_title(f"{feat} (PSI={psi:.3f})")
    ax.legend()

plt.tight_layout()
plt.savefig("drift_distributions.png", dpi=100, bbox_inches="tight")
plt.show()
print("Plot saved to drift_distributions.png")

---

## 3. Drift Detection with Evidently AI

Evidently AI provides production-ready drift reports, data quality checks, and model performance dashboards.

### 3.1 Basic Drift Report

In [ ]:
# Evidently AI drift report
# Requires: pip install evidently

try:
    from evidently.report import Report
    from evidently.metric_preset import DataDriftPreset, DataQualityPreset

    # Create a data drift report
    drift_report = Report(metrics=[DataDriftPreset()])
    drift_report.run(
        reference_data=train_data,
        current_data=prod_with_drift,
    )

    # Save as HTML
    drift_report.save_html("drift_report.html")
    print("Drift report saved to drift_report.html")

    # Get results as dict
    report_dict = drift_report.as_dict()
    for metric in report_dict["metrics"]:
        if "number_of_drifted_columns" in str(metric):
            result = metric.get("result", {})
            print(f"\nDrift Summary:")
            print(f"  Dataset drift detected: {result.get('dataset_drift', 'N/A')}")
            print(f"  Drifted columns: {result.get('number_of_drifted_columns', 'N/A')} / {result.get('number_of_columns', 'N/A')}")

except ImportError:
    print("Evidently is not installed. Install with: pip install evidently")
    print("\nThe report would show:")
    print("  - Per-feature drift detection (KS test, PSI, etc.)")
    print("  - Distribution plots for each feature")
    print("  - Overall dataset drift summary")
    print("  - Statistical test results")

In [ ]:
# Evidently data quality report

try:
    from evidently.report import Report
    from evidently.metric_preset import DataQualityPreset

    quality_report = Report(metrics=[DataQualityPreset()])
    quality_report.run(
        reference_data=train_data,
        current_data=prod_with_drift,
    )

    quality_report.save_html("data_quality_report.html")
    print("Data quality report saved to data_quality_report.html")

except ImportError:
    print("Evidently not installed. The quality report would check:")
    print("  - Missing values per column")
    print("  - Duplicate rows")
    print("  - Constant and near-constant features")
    print("  - Column type mismatches")
    print("  - Range violations")

---

## 4. Production Patterns

### 4.1 Monitoring Architecture

```
                   +------------------+
                   |  ML API Service  |
                   |  (FastAPI)       |
                   +--------+---------+
                            |
              +-------------+-------------+
              |                           |
    +---------v---------+    +-----------v-----------+
    | Prediction Logger |    | Metrics Exporter      |
    | (requests + preds)|    | (Prometheus /metrics)  |
    +---------+---------+    +-----------+-----------+
              |                           |
    +---------v---------+    +-----------v-----------+
    | Feature Store /   |    | Prometheus            |
    | Data Warehouse    |    | (time series DB)      |
    +---------+---------+    +-----------+-----------+
              |                           |
    +---------v---------+    +-----------v-----------+
    | Drift Detection   |    | Grafana               |
    | (batch job)       |    | (dashboards + alerts) |
    +-------------------+    +-----------------------+
```

### 4.2 Key Production Patterns

**Shadow Mode / Canary Deployment**
- Deploy the new model alongside the old one
- Route a small percentage of traffic to the new model
- Compare performance before full rollout

**Champion/Challenger**
- The current production model is the "champion"
- New models are "challengers" that must beat the champion on key metrics
- Automated promotion when the challenger wins

**Feature Store Pattern**
- Centralize feature computation to ensure training/serving consistency
- Features computed once and shared across models
- Prevents training-serving skew

**Circuit Breaker**
- If the model returns errors or drift exceeds a threshold, fall back to a simpler model or rule-based system
- Prevents cascading failures

---

## 5. MLOps in Low-Resource Environments

Not every team has a dedicated ML platform. Here is how to apply MLOps principles with minimal infrastructure.

### Lightweight MLOps Stack

| Full Stack | Lightweight Alternative |
|-----------|------------------------|
| Kubernetes + KubeFlow | Docker Compose on a single server |
| AWS SageMaker | MLflow + FastAPI on a VM |
| Feast (Feature Store) | Parquet files + DVC |
| Airflow/Prefect | Cron jobs + Make targets |
| Grafana + Prometheus | Evidently HTML reports + email alerts |
| S3/GCS | Local NAS or university storage |
| GPU cluster | Google Colab / Kaggle notebooks |

### Minimum Viable MLOps

Even with zero budget, you should have:

1. **Version control** -- Git for code, DVC for data (free)
2. **Experiment tracking** -- MLflow with SQLite backend (free)
3. **Testing** -- pytest in GitHub Actions (free for public repos)
4. **Containerization** -- Docker for reproducible environments (free)
5. **Monitoring** -- Evidently reports generated on a schedule (free)

### Practical Tips

- **Start simple**: A cron job that runs `dvc repro && python report.py` is better than no automation
- **Use Makefiles**: Document commands in a Makefile so anyone can reproduce your work
- **Log everything**: Even `print()` to a file is better than nothing
- **Automate what hurts**: If you do it manually more than twice, automate it

```makefile
# Makefile -- Minimum Viable MLOps

.PHONY: data train evaluate test serve report

data:
	dvc pull
	python src/data/make_dataset.py

train:
	python src/models/train.py

evaluate:
	python src/evaluation/evaluate.py

test:
	pytest tests/ -v --cov=src

serve:
	docker compose up -d --build

report:
	python src/monitoring/drift_report.py

all: data train evaluate test
```

---

## 6. Lab: Drift Detection on Synthetic Data

### Objective

Build a complete drift detection pipeline that:
1. Generates reference and production data
2. Computes PSI and KS tests for all features
3. Generates alerts when drift is detected
4. Visualizes the results

In [ ]:
# Drift monitoring class

class DriftMonitor:
    """
    Monitor data drift between reference and production data.
    Uses PSI and KS test to detect distributional shifts.
    """

    def __init__(self, reference_data: pd.DataFrame, features: list,
                 psi_threshold: float = 0.2, ks_alpha: float = 0.05):
        self.reference_data = reference_data
        self.features = features
        self.psi_threshold = psi_threshold
        self.ks_alpha = ks_alpha
        self.history = []

    def check_drift(self, current_data: pd.DataFrame, batch_id: str = None) -> dict:
        """
        Check for drift in all monitored features.

        Returns a report with per-feature and overall drift status.
        """
        report = {
            "batch_id": batch_id,
            "n_samples": len(current_data),
            "features": {},
            "drifted_features": [],
            "overall_drift": False,
        }

        for feat in self.features:
            ref = self.reference_data[feat].values
            cur = current_data[feat].values

            psi = compute_psi(ref, cur)
            ks = ks_drift_test(ref, cur, alpha=self.ks_alpha)

            feature_report = {
                "psi": round(psi, 4),
                "ks_statistic": round(ks["statistic"], 4),
                "ks_p_value": round(ks["p_value"], 6),
                "psi_drift": psi > self.psi_threshold,
                "ks_drift": ks["drift_detected"],
                "drift": psi > self.psi_threshold or ks["drift_detected"],
            }

            report["features"][feat] = feature_report

            if feature_report["drift"]:
                report["drifted_features"].append(feat)

        report["overall_drift"] = len(report["drifted_features"]) > 0
        self.history.append(report)

        return report

    def print_report(self, report: dict):
        """Pretty-print a drift report."""
        print(f"\nDrift Report (batch={report['batch_id']}, n={report['n_samples']})")
        print("=" * 70)

        for feat, result in report["features"].items():
            status = "DRIFT" if result["drift"] else "OK"
            print(f"  {feat:15s} | PSI={result['psi']:.4f} | "
                  f"KS={result['ks_statistic']:.4f} (p={result['ks_p_value']:.6f}) | "
                  f"[{status}]")

        print(f"\nOverall drift: {report['overall_drift']}")
        if report["drifted_features"]:
            print(f"Drifted features: {report['drifted_features']}")

    def get_history_df(self) -> pd.DataFrame:
        """Return drift history as a DataFrame for tracking over time."""
        rows = []
        for report in self.history:
            for feat, result in report["features"].items():
                rows.append({
                    "batch_id": report["batch_id"],
                    "feature": feat,
                    "psi": result["psi"],
                    "ks_statistic": result["ks_statistic"],
                    "ks_p_value": result["ks_p_value"],
                    "drift": result["drift"],
                })
        return pd.DataFrame(rows)


print("DriftMonitor class defined.")

In [ ]:
# Simulate multiple production batches with increasing drift

np.random.seed(42)

monitor = DriftMonitor(
    reference_data=train_data,
    features=features,
    psi_threshold=0.15,
)

# Simulate 6 weeks of production data with gradual drift
for week in range(1, 7):
    # Gradually shift the distributions
    drift_factor = week * 0.5  # increasing drift

    batch = pd.DataFrame({
        "age": np.random.normal(35 + drift_factor * 2, 10, 500).clip(18, 80),
        "income": np.random.lognormal(10.5 + drift_factor * 0.05, 0.5, 500),
        "credit_score": np.random.normal(680 - drift_factor * 5, 50, 500).clip(300, 850),
        "loan_amount": np.random.lognormal(9.5 + drift_factor * 0.1, 0.6, 500),
    })

    report = monitor.check_drift(batch, batch_id=f"week-{week}")
    monitor.print_report(report)

In [ ]:
# Visualize drift over time

history_df = monitor.get_history_df()

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# PSI over time
for feat in features:
    feat_data = history_df[history_df["feature"] == feat]
    axes[0].plot(feat_data["batch_id"], feat_data["psi"], marker="o", label=feat)

axes[0].axhline(y=0.1, color="orange", linestyle="--", label="Warning threshold")
axes[0].axhline(y=0.2, color="red", linestyle="--", label="Alert threshold")
axes[0].set_title("PSI Over Time")
axes[0].set_xlabel("Batch")
axes[0].set_ylabel("PSI")
axes[0].legend(fontsize=8)
axes[0].tick_params(axis="x", rotation=45)

# KS statistic over time
for feat in features:
    feat_data = history_df[history_df["feature"] == feat]
    axes[1].plot(feat_data["batch_id"], feat_data["ks_statistic"], marker="s", label=feat)

axes[1].set_title("KS Statistic Over Time")
axes[1].set_xlabel("Batch")
axes[1].set_ylabel("KS Statistic")
axes[1].legend(fontsize=8)
axes[1].tick_params(axis="x", rotation=45)

plt.tight_layout()
plt.savefig("drift_over_time.png", dpi=100, bbox_inches="tight")
plt.show()
print("Plot saved to drift_over_time.png")

In [ ]:
# Alerting function -- integrate with your notification system

def check_and_alert(monitor: DriftMonitor, current_data: pd.DataFrame, batch_id: str):
    """
    Run drift check and generate alerts if drift is detected.
    In production, this would send emails, Slack messages, or PagerDuty alerts.
    """
    report = monitor.check_drift(current_data, batch_id=batch_id)

    if report["overall_drift"]:
        alert_message = (
            f"DATA DRIFT ALERT -- Batch: {batch_id}\n"
            f"Drifted features: {report['drifted_features']}\n"
            f"Details:\n"
        )
        for feat in report["drifted_features"]:
            r = report["features"][feat]
            alert_message += f"  - {feat}: PSI={r['psi']:.4f}, KS p-value={r['ks_p_value']:.6f}\n"

        alert_message += "\nRecommended actions:\n"
        alert_message += "  1. Investigate the data source for changes\n"
        alert_message += "  2. Check upstream data pipelines\n"
        alert_message += "  3. Consider retraining the model\n"

        print(alert_message)

        # In production, you would do:
        # send_slack_alert(alert_message)
        # send_email(alert_message)
        # create_pagerduty_incident(alert_message)
    else:
        print(f"Batch {batch_id}: No drift detected. All features within thresholds.")

    return report


# Test the alerting system
print("--- Testing alert on stable data ---")
check_and_alert(monitor, prod_no_drift, "test-stable")

print("\n--- Testing alert on drifted data ---")
check_and_alert(monitor, prod_with_drift, "test-drifted")

---

## Summary

| Topic | Key Takeaway |
|-------|-------------|
| **Model degradation** | All models degrade in production; monitoring is not optional |
| **PSI** | Measures distributional shift; > 0.2 requires action |
| **KS test** | Nonparametric comparison of two distributions; low p-value means drift |
| **Evidently AI** | Production-ready drift reports and dashboards |
| **Production patterns** | Shadow mode, champion/challenger, circuit breakers |
| **Low-resource MLOps** | Start with Git + DVC + MLflow + pytest + Docker |

### Workshop Complete

Over four days we covered the full MLOps lifecycle:

1. **Day 1**: Project structure, experiment tracking (MLflow, DVC, Hydra)
2. **Day 2**: Containerization and APIs (Docker, FastAPI)
3. **Day 3**: Testing and CI/CD (pytest, Great Expectations, GitHub Actions)
4. **Day 4**: Monitoring and production (drift detection, Evidently, production patterns)

The key message: MLOps is not about using every tool -- it is about applying engineering discipline to machine learning so that your models are reproducible, testable, deployable, and maintainable.